# FISH: package-local Pulsar+Spectral example

Run this notebook from `Example/` or the repository root. The local subset uses extracted pulsar CAL products and one matching raw spectral file. See `README.md` and `input_manifest.json` for provenance.

This example uses the existing scientific algorithms. It runs sequentially, writes intermediate products under `Example/output/`, and does not create or delete input symlinks. RFI flagging and baseline subtraction remain outside this example.


In [ ]:
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "FISH_scripts").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or Example/.")
EXAMPLE = ROOT / "Example"
DATA = EXAMPLE / "data"
OUTPUT = EXAMPLE / "output"
sys.path.insert(0, str(ROOT / "FISH_scripts"))

@contextmanager
def working_directory(path):
    previous = Path.cwd()
    try:
        os.chdir(path)
        yield
    finally:
        os.chdir(previous)


## Observation configuration

The default selects Sun15_56_ra, beam M01, spectral file N_0002, with extracted CAL files 0028–0044. The CAL subset includes 60 seconds of preceding context for smoothing. Truncating earlier calibration history can change smoothing at the subset boundary; this example is not a full-scan reproduction. Imaging parameters follow the original notebook.


In [ ]:
SCANS = ["Sun15_56_ra"]
BEAMS = ["M01"]
TCAL = ROOT / "Tcal" / "CAL.20201014.low.W.fits"
POINTING = {scan: DATA / "pointing" / f"{scan}.xlsx" for scan in SCANS}
CENTER_DEG = [42.156000, 58.474000]
IMSIZE = [10, 10]
CELL_DEG = 1 / 60
REST_FREQ_GHZ = 1.420405751768
VMIN_KMS, VMAX_KMS, DV_KMS = -300, 300, 0.15
OVERWRITE_CUBE = False


## Check all inputs before processing

This example starts after PSR CAL extraction. It requires at least two extracted CAL files and one spectral file. Rerunning calibration overwrites matching intermediate products. Use a separate output directory for other selections.


In [ ]:
inputs = {}
problems = []
if not TCAL.is_file():
    problems.append(f"Missing Tcal: {TCAL}")
for scan in SCANS:
    if not POINTING[scan].is_file():
        problems.append(f"Missing pointing table: {POINTING[scan]}")
    for beam in BEAMS:
        cal = sorted((DATA / "cal" / scan / beam).glob("*_cal.fits"))
        spec = sorted((DATA / "spec" / scan).rglob(f"*{beam}*N*.fits"))
        inputs[scan, beam] = (cal, spec)
        if len(cal) < 2:
            problems.append(f"{scan}/{beam}: need >=2 extracted CAL files; found {len(cal)}")
        if not spec:
            problems.append(f"{scan}/{beam}: no noise-off spectral FITS files (*{beam}*N*.fits)")
if problems:
    raise FileNotFoundError("Input checks failed:\n" + "\n".join(problems))
print(f"Inputs found for {len(inputs)} scan/beam combinations. Confirm time coverage and diode settings before running.")


## Calibrate and reconstruct pointing

This preserves the legacy diode duty-cycle correction, smoothing, and band-median scale transfer. Confirm these are appropriate for your observation. Pointing tables must contain `SysTime`, `SDP_PhaPos_X/Y/Z`, and `SDP_AngleM` in the format expected by `xyz2celestial`.


In [ ]:
from Tsys_cal import gaincal
from apply_cal import applycal
from xyz2celestial import xyz2celestial

calibrated_files = []
for scan in SCANS:
    pointing = pd.read_excel(POINTING[scan], sheet_name=0)
    for beam in BEAMS:
        cal_files, spec_files = inputs[scan, beam]
        work = OUTPUT / scan / beam
        (work / "HighCAL").mkdir(parents=True, exist_ok=True)
        (work / "Calibrated").mkdir(exist_ok=True)
        with working_directory(work):
            callist = work / "cal_files.txt"
            callist.write_text("".join(f"{p}\n" for p in cal_files))
            gaincal(str(callist), outname="Tsys_psr", Tcalfits=str(TCAL),
                    beam=int(beam[1:]), plot=False)
            tsys = list(work.glob("Tsys_psr*.fits"))
            if len(tsys) != 1:
                raise RuntimeError(f"Expected one Tsys product in {work}; found {tsys}")
            for path in spec_files:
                applycal(str(path), str(tsys[0]), restfreq=REST_FREQ_GHZ, plot=False)
                products = list((work / "Calibrated").glob(f"{path.stem}_*_calibrated.fits"))
                if len(products) != 1:
                    raise RuntimeError(f"Expected one calibrated product for {path}; found {products}")
                xyz2celestial(str(products[0]), pointing, beam=int(beam[1:]), plot=False)
                calibrated_files.append(products[0].with_name(products[0].stem + "_icrs.fits"))


## Grid the selected observations

Only products explicitly collected above are included. The existing gridder applies one Doppler correction per spectral file. Its spectral WCS currently has a known one-channel offset (`CRPIX3=0`); review/fix this before scientific use of the output cube.


In [ ]:
from gridding_with_doppler_tracking import gridding_with_doppler_tracking

if not calibrated_files:
    raise RuntimeError("No calibrated spectra collected.")
gridlist = OUTPUT / "grid_files.txt"
gridlist.write_text("".join(f"{p}\n" for p in calibrated_files))
cube = OUTPUT / "example_otf_doppler.fits"
gridding_with_doppler_tracking(
    speclist=str(gridlist), center=CENTER_DEG, imsize=IMSIZE,
    cell=CELL_DEG, rest_freq=REST_FREQ_GHZ,
    vmin=VMIN_KMS, vmax=VMAX_KMS, dv=DV_KMS,
    p=[0, 1], outname=str(cube), overwrite=OVERWRITE_CUBE,
)
print(cube)


# Original observation workflow (reference)

The original notebook is preserved below as documentation, including its optional wideband continuum workflow. These historical snippets use remote paths and are not executable cells. Use the portable cells above to run the example.


#### In this document, we show how to use the FISH (FAST InterSterllar HI) scripts to calibrate the FAST CRAFTS data.

### Import necessary packages

### Original code cell 3

```python
import os
import numpy as np
import sys
import gc
import multiprocessing as mp
sys.path.append('/bb8/lingrui/my_projects/FAST_pipeline/FISH_scripts/')
```


### Create soft links to the psr data and and spec data

### Original code cell 5

```python
os.system('rm org_psr org_spec')
os.system('ln -s /bb8/lingrui/FAST_data_all_softlinks/PT2021_0055_1/ org_psr')
os.system('ln -s /bb8/lingrui/FAST_data_all_softlinks/PT2021_0055_2/ org_spec')
```


### Prepare the output directories

### Original code cell 7

```python
Scan_name = ['Sun15_56_ra','Sun15_56_dec']
```


### Original code cell 8

```python
Beam_name = ['M01','M02','M03','M04','M05','M06','M07','M08','M09','M10','M11','M12','M13','M14','M15','M16','M17','M18','M19']
```


### Original code cell 9

```python
# ## run only once
# for i in Scan_name:
#     os.system('rm -rf {}'.format(i))      
#     os.system('mkdir {}'.format(i))
#     for j in Beam_name:
#         os.system('mkdir {}/{}'.format(i,j))
#         os.system('mkdir {}/{}/HighCAL'.format(i,j))
#         os.system('mkdir {}/{}/Calibrated'.format(i,j))
```


#### Generate PSR file list

### Original code cell 11

```python
rootwd = os.getcwd()

for i in Scan_name:
    for j in Beam_name:
        os.system('ls $PWD/org_psr/{0}/*/*{1}*.fits > {0}/{1}/mypsrlist_{0}_{1}'.format(i,j))
```


#### Extract cal (Noise_On and Noise_Off)

### Original code cell 13

```python
from extract_crafts_cal import extract_crafts_cal

rootwd = os.getcwd()

def target_extract_crafts_cal(i,j):
    os.chdir(rootwd+'/'+'{0}/{1}'.format(i,j))
    #print('Dealing with mypsrlist_{0}_{1}'.format(i,j))
    psrlist = np.loadtxt('mypsrlist_{0}_{1}'.format(i,j),dtype=str)
    for psrfits in psrlist:
        extract_crafts_cal(psrfits)

def run_process(i):
    process = []
    for j in Beam_name:
        process.append(mp.Process(target=target_extract_crafts_cal,args=(i,j)))
    [p.start() for p in process]
    [p.join()  for p in process]

for i in Scan_name:
    run_process(i)
```


#### Generate the cal_list

### Original code cell 15

```python
for i in Scan_name:
    for j in Beam_name:
        os.system('ls $PWD/{0}/{1}/HighCAL/* > {0}/{1}/mycallist_{0}_{1}'.format(i,j))
```


#### Generate Tsys_psr


### Original code cell 17

```python
from Tsys_cal import gaincal
!pwd
```


### Original code cell 18

```python
Tcalfits = '/bb8/lingrui/my_projects/FAST_pipeline/FISH_scripts/Tcal/20201014/CAL.20201014.low.W.fits'
rootwd = os.getcwd()

def target_gaincal(i,j):
    os.chdir(rootwd+'/'+'{0}/{1}'.format(i,j))
    beam = int(j.split('M')[-1])
    print('Dealing with mycallist_{0}_{1}'.format(i,j))
    gaincal(callist='mycallist_{0}_{1}'.format(i,j),outname='Tsys_psr_{0}_{1}'.format(i,j),Tcalfits=Tcalfits,beam=beam,plot=True)
    print('Tsys_psr_{0}_{1} Created!'.format(i,j))
    gc.collect()

def run_process(j):
    process = []
    for i in Scan_name:
        process.append(mp.Process(target=target_gaincal,args=(i,j)))
    [p.start() for p in process]
    [p.join()  for p in process]

for j in Beam_name:
    run_process(j)

os.chdir(rootwd)
```


#### Generate speclist

### Original code cell 20

```python
for i in Scan_name:
    for j in Beam_name:
        os.system('ls $PWD/org_spec/{0}/*/*{1}*N* > {0}/{1}/myspeclist_N_{0}_{1}'.format(i,j))
```


#### Apply cal

### Original code cell 22

```python
from apply_cal import applycal
import numpy
import glob

rootwd = os.getcwd()

def target_applycal(i,j):
    os.chdir(rootwd+'/'+'{}/{}'.format(i,j))
    print('Dealing with myspeclist_N_{}_{}'.format(i,j))
    tsysfits = glob.glob('Tsys_psr*.fits')[0]
    speclist = np.loadtxt('myspeclist_N_{}_{}'.format(i,j),dtype=str)
    for specfits in speclist:
        applycal(specfits,tsysfits,restfreq=1.420405751768,plot=True)
        print('Calibrated file of {} Created!'.format(specfits))
    gc.collect()

def run_process(i):
    process = []
    for j in Beam_name:
        process.append(mp.Process(target=target_applycal,args=(i,j)))
    [p.start() for p in process]
    [p.join()  for p in process]

for i in Scan_name:
    run_process(i)
```


#### xyz2radec

### Original code cell 24

```python
for i in Scan_name:
    for j in Beam_name:
        os.system('ls $PWD/{0}/{1}/Calibrated/*calibrated.fits > {0}/{1}/myspeclist_N_{0}_{1}_calibrated'.format(i,j))

from xyz2celestial import xyz2celestial
import glob
import pandas as pd

os.system('ln -s /bb8/lingrui/FAST_data_all_softlinks/PT2021_0055KY PT2021_0055_KYKY')
```


### Original code cell 25

```python
rootwd = os.getcwd()

def target_xyz2celestial(i,j,xyztable):
    os.chdir(rootwd+'/'+'{}/{}'.format(i,j))
    print('Dealing with myspeclist_N_{0}_{1}_calibrated'.format(i,j))
    beam = int(j.split('M')[-1])
    speclist = np.loadtxt('myspeclist_N_{0}_{1}_calibrated'.format(i,j),dtype=str)
    for specfits in speclist:
        xyz2celestial(specfits,xyztable,beam=beam,plot=False)

def run_process(i,xyztable):
    process = []
    for j in Beam_name:
        process.append(mp.Process(target=target_xyz2celestial,args=(i,j,xyztable)))
    [p.start() for p in process]
    [p.join()  for p in process]

for i in Scan_name:
    table = glob.glob(rootwd+'/*KY/'+'{}*.xlsx'.format(i))[0]
    xyztable = pd.read_excel(table,sheet_name=0)
    run_process(i,xyztable)
```


#### Ta to Tmb

### Original code cell 27

```python
# for i in Scan_name:
#     for j in Beam_name:
#         os.system('ls $PWD/{0}/{1}/Calibrated/*calibrated_icrs.fits > {0}/{1}/myspeclist_N_{0}_{1}_calibrated_icrs'.format(i,j))
```


### Original code cell 28

```python
# from Ta2Tmb import Ta2Tmb
# import glob
# import pandas as pd
```


### Original code cell 29

```python
# help(Ta2Tmb)
```


### Original code cell 30

```python
# rootwd = os.getcwd()

# def target_Ta2Tmb(i,j):
#     os.chdir(rootwd+'/'+'{}/{}'.format(i,j))
#     print('Dealing with myspeclist_N_{0}_{1}_calibrated_icrs'.format(i,j))
#     beam = int(j.split('M')[-1])
#     speclist = np.loadtxt('myspeclist_N_{0}_{1}_calibrated_icrs'.format(i,j),dtype=str)
#     for specfits in speclist:
#         Ta2Tmb(specfits,beam=beam,restfreq=1420.405751768)

# def run_process(i):
#     process = []
#     for j in Beam_name:
#         process.append(mp.Process(target=target_Ta2Tmb,args=(i,j)))
#     [p.start() for p in process]
#     [p.join()  for p in process]

# for i in Scan_name:
#     run_process(i)
```


#### Gridding 

### Original code cell 32

```python
# os.system('ls $PWD/Sun17_A_00/*/Calibrated/*_calibrated_icrs.fits \
#               $PWD/Sun17_A_01/*/Calibrated/*_calibrated_icrs.fits \
#               $PWD/Sun17_A_02/*/Calibrated/*_calibrated_icrs.fits \
#               $PWD/Sun17_A_03/*/Calibrated/*_calibrated_icrs.fits \
#               $PWD/Sun17_A_04/*/Calibrated/*_calibrated_icrs.fits > mygridlist')

os.system('ls $PWD/*/*/Calibrated/*_calibrated_icrs.fits > mygridlist_all')
```


### Original code cell 33

```python
from gridding_with_doppler_tracking import gridding_with_doppler_tracking

speclist = 'mygridlist_all'
center = [42.156000,58.474000]
outname = 'Sun15_56_otf_doppler.fits'
gridding_with_doppler_tracking(speclist=speclist, center=center, imsize=[10, 10], cell=1/60, rest_freq=1.420405751768, vmin=-300, vmax=300, dv=0.15, p=[0,1], outname=outname, overwrite=True)
```


# Wideband continuum

#### xyz2radec

### Original code cell 36

```python
# from xyz2celestial import xyz2celestial
# import glob
# import pandas as pd

# os.system('ln -s /TheForce/fast_data/PT2021_0055_KY PT2021_0055_KY')
```


### Original code cell 37

```python
# rootwd = os.getcwd()

# def target_xyz2celestial(i,j,xyztable):
#     os.chdir(rootwd+'/'+'{}/{}'.format(i,j))
#     beam = int(j.split('M')[-1])
#     specfits = glob.glob('Tsys_psr*.fits')[0]
#     print('Dealing with '+specfits)
#     xyz2celestial(specfits,xyztable,beam=beam,plot=False,psrflag=True)

# def run_process(i,xyztable):
#     process = []
#     for j in Beam_name:
#         process.append(mp.Process(target=target_xyz2celestial,args=(i,j,xyztable)))
#     [p.start() for p in process]
#     [p.join()  for p in process]

# for i in Scan_name:
#     table = glob.glob(rootwd+'/*KY/'+'{}*.xlsx'.format(i))[0]
#     xyztable = pd.read_excel(table,sheet_name=0)
#     run_process(i,xyztable)
```


#### gridding

### Original code cell 39

```python
# os.system('ls $PWD/*/*/Tsys_psr*icrs.fits > mygridcont_list')
```


### Original code cell 40

```python
# from gridding_with_doppler_tracking import gridding_with_doppler_tracking

# speclist = 'mygridcont_list'
# center = [284.5,6.5]
# outname = 'Sun17_A_scans_all_doppler_cont.fits'
# vmin = (1.4204-1.45)/1.4204*3e5
# vmax = (1.4204-1.05)/1.4204*3e5
# dv   = 500e-3/1024/1.4204*3e5 
# gridding_with_doppler_tracking(speclist=speclist, center=center, imsize=[360, 360], \
#                                cell=1/60, rest_freq=1.420405751768, vmin=vmin, vmax=vmax, \
#                                dv=dv, p=0,  outname=outname, psrflag=True, overwrite=False)
```
